# GEDI, BIOMASS, and NISAR: Visualization for an AOI

**Date:** March 2nd, 2026

**Author**: Harshini Girish(UAH), Alex Mandel(Development Seed), Henry Rodman(Development Seed), Chuck Daniels(Development Seed)


**Description**:This notebook demonstrates a lightweight, map-first workflow for visualizing where three Earth observation datasets overlap for a given Area of Interest (AOI). We begin by importing required libraries, loading the AOI geometry (sites polygon layer), ensuring it is in WGS84 (EPSG:4326), and computing both a tight AOI bounding box and a buffered bounding box for data discovery. Using a single STAC helper function, we then perform bounding-box searches for NISAR, ESA BIOMASS, and GEDI collections from their respective STAC catalogs and convert the returned STAC items into GeoDataFrames for mapping. Finally, we render an interactive Folium map that first previews the AOI polygon and then overlays the three datasets as separate, color-coded layers with a layer control, enabling quick visual inspection of coverage and overlap before any downstream subsetting or analysis.

## Run This Notebook

To access and run this tutorial within MAAP’s Algorithm Development Environment (ADE), please refer to the [Getting started with the MAAP](#) section of our documentation.

**Disclaimer**: It is highly recommended to run this tutorial within MAAP’s ADE, which already includes packages specific to MAAP, such as maap-py. Running the tutorial outside of the MAAP ADE may lead to errors.


## Install/Import Packages
 
Let's install and load the packages necessary for this tutorial.

In [18]:
import json
from shapely.geometry import box, mapping, Point
import pandas as pd
import geopandas as gpd
import folium

import pystac_client
from pystac import ItemCollection

## Inputs  
Edit these paths/parameters to match your AOI and the collections you want to check.

In [23]:
# --- AOI / Sites input (KML/GeoJSON/Shapefile/etc.) ---
SITES_PATH = "amacayacu_100m.kml"   # change if needed
SITE_ID_COL = None  # set to an existing column name, or leave None to auto-create site_id

# --- STAC endpoints ---
NISAR_STAC_URL   = "https://cmr.earthdata.nasa.gov/stac/ASF"
BIOMASS_STAC_URL = "https://catalog.maap.eo.esa.int/catalogue/"
GEDI_STAC_URL    = "https://cmr.earthdata.nasa.gov/stac/ORNL_CLOUD"

# --- Collections (use exact STAC collection IDs) ---
NISAR_COLLECTION = "NISAR_L2_GCOV_BETA_V1_1"
BIOMASS_COLLECTIONS = ["BiomassLevel1b"]
GEDI_COLLECTION = "GEDI_L4A_AGB_Density_V2_1_2056_2.1"

# --- Optional datetime filters (ISO8601 interval), or None ---
NISAR_DT = None
BIOMASS_DT = None
GEDI_DT = None

# --- Buffer for discovery/search (degrees; EPSG:4326 lon/lat) ---
BBOX_BUFFER_DEG = 0.25



## Load AOI / sites and compute bbox  
We read the sites file into a GeoDataFrame (EPSG:4326), ensure a `site_id`, and compute the AOI bbox for catalog searches.

In [6]:
sites_gdf = gpd.read_file(SITES_PATH)

# Ensure WGS84 lon/lat
if sites_gdf.crs is None:
    sites_gdf = sites_gdf.set_crs("EPSG:4326")
else:
    sites_gdf = sites_gdf.to_crs("EPSG:4326")

# Ensure a site_id column
if SITE_ID_COL and SITE_ID_COL in sites_gdf.columns:
    sites_gdf = sites_gdf.rename(columns={SITE_ID_COL: "site_id"})
elif "site_id" not in sites_gdf.columns:
    sites_gdf["site_id"] = [f"site_{i+1}" for i in range(len(sites_gdf))]

aoi_geom = sites_gdf.geometry.union_all()
minx, miny, maxx, maxy = aoi_geom.bounds
BBOX = (minx, miny, maxx, maxy)

# buffered bbox for discovery
buf = BBOX_BUFFER_DEG
BBOX_BUF = (minx-buf, miny-buf, maxx+buf, maxy+buf)

print("AOI bbox:", BBOX)
print("Buffered bbox:", BBOX_BUF)
sites_gdf[["site_id", "geometry"]].head()


AOI bbox: (-70.2926180592577, -3.81826995097037, -70.2404260082027, -3.78652050791343)
Buffered bbox: (-70.5426180592577, -4.06826995097037, -69.9904260082027, -3.53652050791343)


,site_id,geometry
0,site_1,"MULTIPOLYGON (((-70.29262 -3.7921, -70.29262 -..."


## AOI preview map
Visualize the AOI polygons before running any STAC searches.

In [24]:
# Quick AOI preview map (sites + AOI outline)

aoi_map = folium.Map(
    location=[(miny + maxy) / 2, (minx + maxx) / 2],
    zoom_start=10,
    tiles="OpenStreetMap",
)

# Site polygons (purple)
folium.GeoJson(
    sites_gdf,
    name="Sites",
    style_function=lambda _: {"color": "purple", "weight": 2, "fillOpacity": 0.10},
).add_to(aoi_map)

# AOI outline (red)
folium.GeoJson(
    data={"type": "Feature", "geometry": mapping(aoi_geom), "properties": {"name": "AOI"}},
    name="AOI",
    style_function=lambda _: {"color": "red", "weight": 3, "fillOpacity": 0.0},
).add_to(aoi_map)

folium.LayerControl(collapsed=False).add_to(aoi_map)
aoi_map


## STAC helper

Single helper to run a STAC bbox search and directly return a GeoDataFrame for mapping.

In [25]:
def stac_search_to_gdf(catalog_url, collections, bbox, datetime=None, limit=200):
    """Run a STAC bbox search and return results as a GeoDataFrame (EPSG:4326)."""
    cat = pystac_client.Client.open(catalog_url)
    search = cat.search(collections=collections, bbox=bbox, datetime=datetime, max_items=limit)
    items = list(search.items())
    return gpd.GeoDataFrame.from_features([it.to_dict() for it in items], crs="EPSG:4326")



## NISAR coverage
Search the NISAR STAC catalog for the chosen NISAR collection intersecting the AOI bbox.

In [26]:
nisar_gdf = stac_search_to_gdf(
    catalog_url=NISAR_STAC_URL,
    collections=[NISAR_COLLECTION],
    bbox=BBOX,
    datetime=NISAR_DT,
    limit=500,
)
print(f"NISAR items found for {NISAR_COLLECTION}: {len(nisar_gdf)}")
nisar_gdf.head()



NISAR items found for NISAR_L2_GCOV_BETA_V1_1: 9


,geometry,datetime,storage:schemes,start_datetime,end_datetime
0,"POLYGON ((-69.99474 -3.94381, -70.45989 -1.795...",2025-11-06T10:27:57Z,"{'aws': {'type': 'aws-s3', 'platform': 'https:...",2025-11-06T10:27:57.000Z,2025-11-06T10:28:32.999Z
1,"POLYGON ((-67.8897 -3.94315, -68.34109 -1.8547...",2025-11-11T10:19:38Z,"{'aws': {'type': 'aws-s3', 'platform': 'https:...",2025-11-11T10:19:38.000Z,2025-11-11T10:20:12.999Z
2,"POLYGON ((-70.35039 -1.73374, -70.81558 -3.881...",2025-11-30T22:58:08Z,"{'aws': {'type': 'aws-s3', 'platform': 'https:...",2025-11-30T22:58:08.000Z,2025-11-30T22:58:43.999Z
3,"POLYGON ((-67.89068 -3.93366, -68.34215 -1.845...",2025-12-05T10:19:39Z,"{'aws': {'type': 'aws-s3', 'platform': 'https:...",2025-12-05T10:19:39.000Z,2025-12-05T10:20:13.999Z
4,"POLYGON ((-69.98483 -3.99241, -70.4499 -1.8445...",2025-12-12T10:27:58Z,"{'aws': {'type': 'aws-s3', 'platform': 'https:...",2025-12-12T10:27:58.000Z,2025-12-12T10:28:33.999Z


## BIOMASS coverage
Search the ESA BIOMASS STAC catalog for the chosen BIOMASS collection(s) intersecting the AOI bbox.

In [27]:
biomass_gdf = stac_search_to_gdf(
    catalog_url=BIOMASS_STAC_URL,
    collections=BIOMASS_COLLECTIONS,
    bbox=BBOX,
    datetime=BIOMASS_DT,
    limit=500,
)
print(f"BIOMASS items found ({BIOMASS_COLLECTIONS}): {len(biomass_gdf)}")
biomass_gdf.head()


BIOMASS items found (['BiomassLevel1b']): 7


,geometry,eofeos:repeat_cycle_id,start_datetime,end_datetime,processing:facility,product:type,eofeos:global_coverage_id,sat:anx_datetime,title,platform,...,sar:observation_direction,sar:polarizations,auth:schemes,sar:instrument_mode,eofeos:orbit_drift_flag,processing:datetime,eofeos:mission_phase,updated,sat:absolute_orbit,product:acquisition_type
0,"POLYGON ((-70.63902 -4.03723, -70.09614 -4.150...",4,2026-02-20T22:49:07.442Z,2026-02-20T22:49:27.731Z,Biomass CPF,S1_DGM__1S,1,2026-02-20T21:59:28.073Z,BIO_S1_DGM__1S_20260220T224907_20260220T224927...,Biomass,...,left,"[HH, HV, VH, VV]","{'s3': {'type': 's3'}, 'oidc': {'openIdConnect...",SM,False,2026-02-21T10:04:09Z,TOMOGRAPHIC,2026-02-27T14:23:39Z,4365,nominal
1,"POLYGON ((-70.64701 -4.03701, -70.10399 -4.150...",5,2026-02-23T22:49:09.093Z,2026-02-23T22:49:29.382Z,Biomass CPF,S1_DGM__1S,1,2026-02-23T21:59:29.731Z,BIO_S1_DGM__1S_20260223T224909_20260223T224929...,Biomass,...,left,"[HH, HV, VH, VV]","{'s3': {'type': 's3'}, 'oidc': {'openIdConnect...",SM,False,2026-02-24T11:00:18Z,TOMOGRAPHIC,2026-02-27T14:28:20Z,4409,nominal
2,"POLYGON ((-70.66071 -4.03709, -70.11783 -4.150...",7,2026-03-01T22:49:12.783Z,2026-03-01T22:49:33.071Z,Biomass CPF,S1_DGM__1S,1,2026-03-01T21:59:33.396Z,BIO_S1_DGM__1S_20260301T224912_20260301T224933...,Biomass,...,left,"[HH, HV, VH, VV]","{'s3': {'type': 's3'}, 'oidc': {'openIdConnect...",SM,False,2026-03-02T11:02:12Z,TOMOGRAPHIC,2026-03-02T13:01:34Z,4497,nominal
3,"POLYGON ((-70.62412 -4.03726, -70.08149 -4.150...",2,2026-02-14T22:49:03.840Z,2026-02-14T22:49:24.128Z,Biomass CPF,S1_DGM__1S,1,2026-02-14T21:59:24.479Z,BIO_S1_DGM__1S_20260214T224903_20260214T224924...,Biomass,...,left,"[HH, HV, VH, VV]","{'s3': {'type': 's3'}, 'oidc': {'openIdConnect...",SM,False,2026-02-15T10:26:08Z,TOMOGRAPHIC,2026-02-27T14:14:10Z,4277,nominal
4,"POLYGON ((-70.6162 -4.03741, -70.07372 -4.1507...",1,2026-02-11T22:49:02.000Z,2026-02-11T22:49:22.289Z,Biomass CPF,S1_DGM__1S,1,2026-02-11T21:59:22.648Z,BIO_S1_DGM__1S_20260211T224902_20260211T224922...,Biomass,...,left,"[HH, HV, VH, VV]","{'s3': {'type': 's3'}, 'oidc': {'openIdConnect...",SM,False,2026-02-12T10:02:06Z,TOMOGRAPHIC,2026-02-27T14:09:30Z,4233,nominal


## GEDI coverage
Search the GEDI STAC catalog for GEDI L4A AGBD intersecting the buffered AOI bbox (GEDI tracks can be sparse).

In [28]:
gedi_gdf = stac_search_to_gdf(
    catalog_url=GEDI_STAC_URL,
    collections=[GEDI_COLLECTION],
    bbox=BBOX_BUF,      # use buffered bbox for GEDI discovery
    datetime=GEDI_DT,
    limit=500,
)
print(f"GEDI items found for {GEDI_COLLECTION}: {len(gedi_gdf)}")
gedi_gdf.head()


GEDI items found for GEDI_L4A_AGB_Density_V2_1_2056_2.1: 138


,geometry,datetime,storage:schemes,start_datetime,end_datetime
0,"POLYGON ((-90.92339 -28.63784, -88.13659 -25.8...",2019-04-20T09:42:53Z,"{'aws': {'type': 'aws-s3', 'platform': 'https:...",2019-04-20T09:42:53.000Z,2019-04-20T09:52:33.000Z
1,"POLYGON ((-72.56963 -0.28696, -70.40899 -3.350...",2019-05-17T11:26:30Z,"{'aws': {'type': 'aws-s3', 'platform': 'https:...",2019-05-17T11:26:30.000Z,2019-05-17T11:36:36.000Z
2,"POLYGON ((-73.19654 -0.02675, -71.0381 -3.0880...",2019-05-24T08:44:16Z,"{'aws': {'type': 'aws-s3', 'platform': 'https:...",2019-05-24T08:44:16.000Z,2019-05-24T08:54:31.000Z
3,"POLYGON ((-90.70751 -29.22035, -87.8886 -26.44...",2019-06-12T12:34:08Z,"{'aws': {'type': 'aws-s3', 'platform': 'https:...",2019-06-12T12:34:08.000Z,2019-06-12T12:44:00.000Z
4,"POLYGON ((-72.85289 -0.06564, -70.69566 -3.124...",2019-06-18T22:34:32Z,"{'aws': {'type': 'aws-s3', 'platform': 'https:...",2019-06-18T22:34:32.000Z,2019-06-18T22:44:43.000Z


## Map: AOI + NISAR + BIOMASS + GEDI
Interactive map showing all three datasets as different colored layers.

In [32]:
# One map with AOI + NISAR + BIOMASS + GEDI (different colors)

m = folium.Map(location=[(miny + maxy) / 2, (minx + maxx) / 2], zoom_start=8, tiles="OpenStreetMap")

def _style(color, weight=2, fill_opacity=0.05):
    return lambda _: {"color": color, "weight": weight, "fillOpacity": fill_opacity}

# AOI outline (red)
folium.GeoJson(
    data={"type": "Feature", "geometry": mapping(aoi_geom), "properties": {"name": "AOI"}},
    name="AOI",
    style_function=_style("red", weight=3, fill_opacity=0.0),
).add_to(m)

# NISAR footprints (blue)
if len(nisar_gdf) > 0:
    folium.GeoJson(
        nisar_gdf,
        name="NISAR",
        style_function=_style("blue", weight=2, fill_opacity=0.05),
    ).add_to(m)

# BIOMASS footprints (green)
if len(biomass_gdf) > 0:
    folium.GeoJson(
        biomass_gdf,
        name="BIOMASS",
        style_function=_style("green", weight=2, fill_opacity=0.05),
    ).add_to(m)

# GEDI footprints (orange)
if len(gedi_gdf) > 0:
    folium.GeoJson(
        gedi_gdf,
        name="GEDI L4A AGBD",
        style_function=_style("orange", weight=1, fill_opacity=0.001),
    ).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m